# 03. Histogramas y transformaciones de intensidad

**Objetivo:** describir la distribución de intensidades de una imagen y aprender a modificarla de forma segura.

In [ ]:
import numpy as np

from filtrado_digital.io import cargar_imagen, a_grises, ruta_imagen_ejemplo
from filtrado_digital.visualizacion import comparar, imagen_e_histograma
from filtrado_digital.histogramas import histograma_manual, histograma_numpy
from filtrado_digital.transformaciones import negativo, raiz_cuadrada, logaritmica

## 1. Histograma, contraste y rango dinámico

Un **histograma de intensidades** cuenta cuántos píxeles poseen cada valor entre `0` y `255`. El eje horizontal representa la intensidad y el eje vertical la frecuencia.

El **contraste** describe la diferencia entre regiones claras y oscuras. El **rango dinámico**, en este contexto, es el intervalo entre la intensidad mínima y máxima utilizadas por la imagen.

In [ ]:
foto = cargar_imagen(ruta_imagen_ejemplo())
imagen = a_grises(foto)
imagen_e_histograma(imagen, "Fotografía original")

h_manual = histograma_manual(imagen)
h_numpy = histograma_numpy(imagen)
print("¿Coinciden los histogramas?", np.array_equal(h_manual, h_numpy))

## 2. `uint8`, números reales y saturación

Las imágenes de 8 bits suelen almacenarse como `uint8`, con valores enteros de `0` a `255`. Sin embargo, muchas transformaciones incluyen multiplicaciones, divisiones o funciones matemáticas que producen decimales.

Una estrategia segura es:

1. convertir temporalmente a `float32` o `float64`;
2. realizar la operación;
3. limitar el resultado al intervalo `[0, 255]` con `np.clip()`;
4. convertir de nuevo a `uint8`.

Esto evita resultados fuera del rango válido y hace explícita la conversión.

In [ ]:
# Ejemplo de transformación lineal.
datos_float = imagen.astype(np.float32)
transformada_float = 0.6 * datos_float + 40
transformada_lineal = np.clip(transformada_float, 0, 255).astype(np.uint8)

comparar([imagen, transformada_lineal], ["Original", "s = 0.6 r + 40"])
print("Original:", imagen.min(), imagen.max(), imagen.dtype)
print("Transformada:", transformada_lineal.min(), transformada_lineal.max(), transformada_lineal.dtype)

## 3. Transformaciones puntuales

Una **transformación puntual** calcula cada píxel de salida usando solamente la intensidad del píxel correspondiente, sin consultar a sus vecinos.

### Negativo

Para una imagen de 8 bits: `s = 255 - r`.

In [ ]:
img_neg = negativo(imagen)
imagen_e_histograma(img_neg, "Negativo")

### Raíz cuadrada y logaritmo

Estas transformaciones no lineales redistribuyen las intensidades. Después del cálculo se normalizan para volver al intervalo `[0, 255]`.

In [ ]:
img_raiz = raiz_cuadrada(imagen)
img_log = logaritmica(imagen)
comparar([imagen, img_raiz, img_log], ["Original", "Raíz", "Logarítmica"])

## Conclusiones

- El histograma resume cómo se distribuyen las intensidades.
- Conviene realizar operaciones matemáticas en punto flotante y convertir a `uint8` al final.
- `np.clip()` permite controlar la saturación en `[0, 255]`.
- Las transformaciones puntuales modifican cada píxel sin considerar su vecindad.